# Polygon area and perimeter vs. `boundary_max_deviation_um`

Follow-up to the `simplify_tol_um` -> `boundary_max_deviation_um` rename
(`src/MERci/acquisition/mosaic.py`'s `segment_mosaic_tissue`): that
parameter is a Shapely `Polygon.simplify()` Douglas-Peucker tolerance, not
a fixed output segment length. This notebook makes that concrete by
resegmenting the same real mosaic canvas at a range of tolerances
(`[0.1, 0.25, 0.5, 1, 2, 3, 4, 6, 8, 10, 14, 16, 18, 20, 22, 24]` um) and
plotting how the traced tissue polygon's own area and total perimeter
respond -- every other `segment_mosaic_tissue` parameter held fixed, so
the sweep isolates this one knob.


## 1 — Setup


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# notebooks/tests/<subfolder>/ is three levels under the repo root (MERci/),
# same convention as before_imaging/{regular,multi_z}/ -- see CLAUDE.md.
MERCI_DIR  = Path(os.getcwd()).parent.parent.parent   # MERci/
SAMPLE_DIR = MERCI_DIR.parent                  # this repo's own sandbox experiment
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.mosaic import segment_mosaic_tissue, load_mosaic_canvas
from MERci.plots.experiment_plots import get_merci_figures_dir

NOTEBOOK_NAME = "sweep_boundary_max_deviation_vs_polygon_area"
CACHE_DIR = SAMPLE_DIR / "analysis" / "cache" / NOTEBOOK_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = get_merci_figures_dir(SAMPLE_DIR, "tests", NOTEBOOK_NAME, subfolder="create_positions")

print(f"MERCI_DIR : {MERCI_DIR}")
print(f"SAMPLE_DIR: {SAMPLE_DIR}")


## 2 — Load the bundled example canvas + baseline segmentation params

Same real, mosaic-derived canvas (`LT066_sample_01/merfish`, bundled as
`MERci/data/mosaic_canvas_examples/lineage_tracing/canvas.npz`) and the
same `SAFE_PARAMS` baseline already vetted in
`01_compare_fov_coverage_constraint.ipynb` (this same directory) --
`boundary_max_deviation_um` is the only parameter swept below.


In [ ]:
canvas = load_mosaic_canvas(MERCI_DIR / "data" / "mosaic_canvas_examples" / "lineage_tracing" / "canvas.npz")
print(f"Canvas: {canvas.image.shape[1]}x{canvas.image.shape[0]} px at {canvas.pixel_size_um:.2f} um/px")

# Same baseline as 01_compare_fov_coverage_constraint.ipynb's SAFE_PARAMS --
# boundary_max_deviation_um is overridden per sweep value below.
BASE_PARAMS = dict(threshold=400, smooth_sigma_um=10.0, close_radius_um=50.0, open_radius_um=15.0,
                    margin_um=25.0, min_tissue_area_um2=4_000_000.0, min_hole_area_um2=12_000.0,
                    min_island_area_um2=50_000.0)

DEVIATION_VALUES_UM = [0.1, 0.25, 0.5, 1, 2, 3, 4, 6, 8, 10, 14, 16, 18, 20, 22, 24]


## 3 — Segment at every `boundary_max_deviation_um`, cache the result


In [ ]:
SWEEP_CSV = CACHE_DIR / "boundary_max_deviation_sweep.csv"

if SWEEP_CSV.exists():
    sweep_df = pd.read_csv(SWEEP_CSV)
    print(f"Loaded cached {SWEEP_CSV} ({len(sweep_df)} rows).")
else:
    rows = []
    for dev_um in DEVIATION_VALUES_UM:
        seg = segment_mosaic_tissue(canvas, boundary_max_deviation_um=dev_um, **BASE_PARAMS)
        tissue_area_um2 = sum(p.area for p in seg.tissue_polygons)
        tissue_perimeter_um = sum(p.exterior.length for p in seg.tissue_polygons)
        hole_area_um2 = sum(p.area for p in seg.hole_polygons)
        n_vertices = sum(len(p.exterior.coords) for p in seg.tissue_polygons)
        rows.append({
            "boundary_max_deviation_um": dev_um,
            "n_tissue_pieces": len(seg.tissue_polygons),
            "tissue_area_um2": tissue_area_um2,
            "tissue_perimeter_um": tissue_perimeter_um,
            "n_holes": len(seg.hole_polygons),
            "hole_area_um2": hole_area_um2,
            "net_area_um2": tissue_area_um2 - hole_area_um2,
            "n_vertices": n_vertices,
        })
        print(f"boundary_max_deviation_um={dev_um:5.2f}: "
              f"tissue_area={tissue_area_um2/1e6:.4f} mm^2, "
              f"tissue_perimeter={tissue_perimeter_um/1e3:.3f} mm, "
              f"n_vertices={n_vertices}, n_holes={len(seg.hole_polygons)}")
    sweep_df = pd.DataFrame(rows)
    sweep_df.to_csv(SWEEP_CSV, index=False)
    print(f"Cached {SWEEP_CSV}.")

sweep_df


## 4 — Polygon area vs. `boundary_max_deviation_um`


In [ ]:
PLOT_TITLE_FONTSIZE, PLOT_LABEL_FONTSIZE = 13, 11
PLOT_TICK_FONTSIZE = 10

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(sweep_df["boundary_max_deviation_um"], sweep_df["tissue_area_um2"] / 1e6, "o-", color="tab:blue")
ax.set_xscale("log")
ax.set_xlabel("boundary_max_deviation_um (log scale)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_ylabel("Tissue polygon area (mm^2)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title("Segmented tissue polygon area vs. simplify() max-deviation tolerance", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(axis="both", labelsize=PLOT_TICK_FONTSIZE)
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "06_boundary_max_deviation_vs_polygon_area.png", dpi=150)
plt.show()
print(f"Saved: {FIG_DIR / '06_boundary_max_deviation_vs_polygon_area.png'}")


## 5 — Total perimeter vs. `boundary_max_deviation_um`


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(sweep_df["boundary_max_deviation_um"], sweep_df["tissue_perimeter_um"] / 1e3, "o-", color="tab:green")
ax.set_xscale("log")
ax.set_xlabel("boundary_max_deviation_um (log scale)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_ylabel("Tissue polygon total perimeter (mm)", fontsize=PLOT_LABEL_FONTSIZE)
ax.set_title("Segmented tissue polygon perimeter vs. simplify() max-deviation tolerance", fontsize=PLOT_TITLE_FONTSIZE)
ax.tick_params(axis="both", labelsize=PLOT_TICK_FONTSIZE)
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "06_boundary_max_deviation_vs_polygon_perimeter.png", dpi=150)
plt.show()
print(f"Saved: {FIG_DIR / '06_boundary_max_deviation_vs_polygon_perimeter.png'}")


## 6 — Takeaways

- **Perimeter shrinks steadily and substantially**: 35.416 mm at
  `boundary_max_deviation_um=0.1-1.0` (flat -- the traced contour already
  has no vertex-to-vertex deviation below ~1um at this canvas resolution)
  down to 32.231 mm at `=24.0` -- an 9.0% drop, monotonic across the whole
  sweep. Vertex count drops in step, 3554 -> 185 (19x).
- **Area barely moves by comparison**: 35.5950 mm^2 down to 35.5688 mm^2,
  a 0.073% change over the same range -- two orders of magnitude smaller
  an effect than the perimeter change.
- **Together these confirm the max-deviation semantics directly**: cutting
  corners off a polygon's boundary (what Douglas-Peucker simplification
  does) shortens its perimeter a lot while leaving the area it encloses
  almost unchanged, exactly as expected for a tolerance bounding how far
  the simplified boundary may deviate from the original traced contour --
  not a fixed output segment length, and an area-preserving operation far
  more than an area-shrinking one.
